In [1]:
using Pkg
Pkg.activate(".")

using SpectralIRLS
using Random, LinearAlgebra

[ Info: Precompiling SpectralIRLS [3ead10a5-a636-4c1d-b4eb-bf43a9e33d59] (cache misses: include_dependency fsize change (2))

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up
  Activating project at `~/codes/Julia/MyPack/SpectralIRLS`


In [8]:
rng = MersenneTwister(3124)

m = 2000
p = 500

X = generate_correlated_design(m, p; rho=0.5, seed=1)
βtrue = randn(rng, p) / sqrt(p)
family = StudentT(4.0, 1.0)
y, _, _ = simulate_y_from_beta(X, βtrue, family; seed=3124, beta0=0.5)

Xfit = hcat(ones(m), X)

2000×501 Matrix{Float64}:
 1.0  -0.136915    0.0421703  -0.478242   …  -1.13512    -1.15466   -0.407964
 1.0   0.960062    1.35851     1.6313         1.43606    -0.45434    1.28203
 1.0  -0.759071   -0.937852    1.08431       -0.0642585  -0.638169  -2.41601
 1.0  -1.33878     0.246184    1.05815       -0.365317   -0.478102   1.89334
 1.0  -2.21615    -0.470457   -0.129865      -0.862001    0.337101  -1.80412
 1.0   1.048       1.06368     0.994464   …  -0.945019   -1.26569   -0.162939
 1.0   0.835791    1.10344    -0.416337       0.50125     0.798112   1.05246
 1.0  -0.0661032  -0.50848     0.49285        1.29982     0.758202   0.554086
 1.0   0.256555    0.395606    0.492779      -0.405239   -0.419452   0.401302
 1.0  -0.633314   -0.393124    1.15141       -0.574702    0.276624   0.392809
 1.0   0.389039   -0.289901   -1.08368    …   0.621964    0.78734    0.134915
 1.0   2.10497    -0.385149   -0.223423       0.634556    0.637671   0.917651
 1.0   0.519751   -1.22737    -0.240509    

In [10]:
include("benchmark/benchmark.jl")

results = benchmark_irls_methods(Xfit, y; family=family, ridge=0.0)

  Activating 

7×8 DataFrame
 Row │ Method       Time_ms   Iter   Loss     GradNorm    RelGradNorm  Converged  LossGap     
     │ String       Float64   Int64  Float64  Float64     Float64      Bool       Float64     
─────┼────────────────────────────────────────────────────────────────────────────────────────
   1 │ Spectral-MM   3208.92     38  1894.97  0.00049444  1.30105e-7        true  8.57199e-11
   2 │ Cholesky     13692.0      23  1894.97  4.18217e-8  1.10048e-11       true  1.13687e-12
   3 │ LSMR         14693.8      56  1894.97  0.0011247   2.95947e-7        true  1.11511e-8
   4 │ CRLS         19462.6      28  1894.97  1.74865e-6  4.60132e-10       true  0.0
   5 │ CG           22237.0      26  1894.97  3.39111e-6  8.9232e-10        true  1.13687e-12
   6 │ CGLS         24737.0      26  1894.97  1.32098e-6  3.47596e-10       true  1.13687e-12
   7 │ LSQR         26187.7      39  1894.97  0.00201811  5.31035e-7        true  1.56053e-8

project at `~/codes/Julia/MyPack/SpectralIRLS`


LoadError: MethodError: no method matching benchmark_irls_methods(::Matrix{Float64}, ::Vector{Float64}; family::SmoothQuantile{Float64}, ridge::Float64)
The function `benchmark_irls_methods` exists, but no method is defined for this combination of argument types.

[0mClosest candidates are:
[0m  benchmark_irls_methods(::Any, ::Any, [91m::IRLSFamily[39m; spectral, inner, outer, ridge, maxiter, gtol, relgtol, outer_nesterov, samples)[91m got unsupported keyword argument "family"[39m
[0m[90m   @[39m [36mMain[39m [90m~/codes/Julia/MyPack/SpectralIRLS/benchmark/[39m[90m[4mbenchmark.jl:44[24m[39m


## Four Examples

In [3]:
using BenchmarkTools,DataFrames,GLM,LinearAlgebra,Random,Statistics
import Distributions

glm_spec(::SpectralIRLS.BernoulliLogit)=(Distributions.Bernoulli(),LogitLink())
glm_spec(::SpectralIRLS.BernoulliProbit)=(Distributions.Bernoulli(),ProbitLink())
glm_spec(::SpectralIRLS.GammaLog)=(Distributions.Gamma(),LogLink())
glm_spec(::SpectralIRLS.GammaInverse)=(Distributions.Gamma(),InverseLink())
glm_spec(::SpectralIRLS.GaussianIdentity)=(Distributions.Normal(),IdentityLink())
glm_spec(::SpectralIRLS.GaussianLog)=(Distributions.Normal(),LogLink())
glm_spec(::SpectralIRLS.PoissonLog)=(Distributions.Poisson(),LogLink())
glm_spec(f::SpectralIRLS.NegativeBinomialLog)=(Distributions.NegativeBinomial(f.theta),LogLink())

glm_supported(::SpectralIRLS.IRLSFamily)=false
glm_supported(::SpectralIRLS.GLMFamily)=true
glm_supported(::SpectralIRLS.TweedieLog)=false
glm_supported(::SpectralIRLS.BinomialLogit)=false

function evaluate_solution(X,y,beta,family;ridge=0.0,w_floor=1e-12,grad_scale=1.0)
    T=eltype(X); m,p=size(X)
    eta=X*beta; mu=zeros(T,m); w=zeros(T,m); r=zeros(T,m); g=zeros(T,p)
    loss=logloss_xb(family,eta,y,beta;ridge=T(ridge))
    grad_weights_xb!(family,g,mu,w,r,X,eta,y,beta;ridge=T(ridge),w_floor=T(w_floor))
    gn=norm(g)
    return loss,gn,gn/grad_scale
end

function initial_grad_scale(X,y,family;ridge=0.0,w_floor=1e-12)
    T=eltype(X); m,p=size(X)
    beta=zeros(T,p); eta=zeros(T,m); mu=zeros(T,m); w=zeros(T,m); r=zeros(T,m); g=zeros(T,p)
    grad_weights_xb!(family,g,mu,w,r,X,eta,y,beta;ridge=T(ridge),w_floor=T(w_floor))
    return one(T)+norm(g)
end

function bench_time(f;samples=5)
    b=@benchmarkable $f() evals=1
    b.params.samples=samples
    return median(run(b)).time/1e6
end

function benchmark_irls_methods(
    X,y,family::IRLSFamily;
    spectral,inner,outer,ridge=0.0,maxiter=1000,gtol=1e-8,relgtol=1e-8,
    outer_nesterov=true,samples=5
)
    T=eltype(X); grad_scale=initial_grad_scale(X,y,family;ridge=ridge); results=NamedTuple[]

    f=()->irls_cholesky(X,y;family=family,ridge=T(ridge),maxiter=maxiter,
        gtol=T(gtol),relgtol=T(relgtol),outer_nesterov=outer_nesterov,verbose=false)
    fit=f(); tm=bench_time(f;samples=samples)
    loss,gn,rgn=evaluate_solution(X,y,fit.beta,family;ridge=ridge,grad_scale=grad_scale)
    push!(results,(Method="Cholesky",Time_ms=tm,Iter=fit.iters,Loss=loss,GradNorm=gn,RelGradNorm=rgn,Converged=fit.converged))

    f=()->spectral_mm(X,y;family=family,spectral=spectral,inner=inner,outer=outer)
    fit=f(); tm=bench_time(f;samples=samples)
    loss,gn,rgn=evaluate_solution(X,y,fit.beta,family;ridge=ridge,grad_scale=grad_scale)
    push!(results,(Method="Spectral-MM",Time_ms=tm,Iter=fit.iters,Loss=loss,GradNorm=gn,RelGradNorm=rgn,Converged=fit.converged))

    for solver in (:cg,:cgls,:crls,:lsqr,:lsmr)
        f=()->irls_krylov(X,y;family=family,solver=solver,ridge=T(ridge),maxiter=maxiter,
            gtol=T(gtol),relgtol=T(relgtol),outer_nesterov=outer_nesterov,verbose=false)
        fit=f(); tm=bench_time(f;samples=samples)
        loss,gn,rgn=evaluate_solution(X,y,fit.beta,family;ridge=ridge,grad_scale=grad_scale)
        push!(results,(Method=uppercase(string(solver)),Time_ms=tm,Iter=fit.iters,Loss=loss,GradNorm=gn,RelGradNorm=rgn,Converged=fit.converged))
    end

    if glm_supported(family) && ridge==0
        D,L=glm_spec(family)
        f=()->glm(X,y,D,L;maxiter=maxiter,atol=gtol,rtol=relgtol)
        fit=f(); tm=bench_time(f;samples=samples)
        loss,gn,rgn=evaluate_solution(X,y,coef(fit),family;ridge=0.0,grad_scale=grad_scale)
        push!(results,(Method="GLM.jl",Time_ms=tm,Iter=missing,Loss=loss,GradNorm=gn,RelGradNorm=rgn,Converged=true))
    end

    df=DataFrame(results)
    df.LossGap=df.Loss.-minimum(df.Loss)
    sort!(df,:Time_ms)
    return df
end


benchmark_irls_methods (generic function with 1 method)

In [13]:
seed=3124
p=5000; m=2p; T=Float64
beta_true=randn(MersenneTwister(seed),T,p)./sqrt(T(p))
beta0_true=T(0.5)
X=generate_correlated_design(m,p;rho=0.5,seed=1,T=T)

# Change only this line to run the four new examples.
family=SmoothQuantile(0.25,0.1)
# family=Expectile(0.25)
# family=PseudoHuber(1.345)
# family=StudentT(4.0,1.0)

y,mu,eta=simulate_y_from_beta(X,beta_true,family;seed=seed,beta0=beta0_true)
Xfit=hcat(ones(T,m),X)

spectral=SpectralOptions{T}(k=5,krylovdim=12,ridge=0.0,resid_tol=5e-1,correction_tol=5e-2)
inner=InnerOptions{T}(solver=:pcg,maxiter=200,eta_max=0.6,forcing_c=10.0,forcing_alpha=0.0,abstol=1e-5,nesterov=true)
outer=OuterOptions{T}(maxiter=200,gtol=1e-6,relgtol=1e-8,nesterov=true,verbose=false)

results=benchmark_irls_methods(Xfit,y,family;spectral=spectral,inner=inner,outer=outer,
    ridge=0.0,maxiter=1000,gtol=1e-6,relgtol=1e-8,outer_nesterov=true,samples=5)
show(results,allrows=true,allcols=true)


7×8 DataFrame
 Row │ Method       Time_ms   Iter   Loss     GradNorm    RelGradNorm  Converged  LossGap     
     │ String       Float64   Int64  Float64  Float64     Float64      Bool       Float64     
─────┼────────────────────────────────────────────────────────────────────────────────────────
   1 │ Spectral-MM   3448.63     38  1894.97  0.00049444  1.30105e-7        true  8.57199e-11
   2 │ Cholesky     13554.5      23  1894.97  4.18217e-8  1.10048e-11       true  1.13687e-12
   3 │ LSMR         14786.7      56  1894.97  0.0011247   2.95947e-7        true  1.11511e-8
   4 │ CRLS         18464.2      28  1894.97  1.74865e-6  4.60132e-10       true  0.0
   5 │ CGLS         18972.0      26  1894.97  1.32098e-6  3.47596e-10       true  1.13687e-12
   6 │ CG           23595.7      26  1894.97  3.39111e-6  8.9232e-10        true  1.13687e-12
   7 │ LSQR         24798.1      39  1894.97  0.00201811  5.31035e-7        true  1.56053e-8

In [7]:
seed=3124
p=5000; m=2p; T=Float64
beta_true=randn(MersenneTwister(seed),T,p)./sqrt(T(p))
beta0_true=T(0.5)
X=generate_correlated_design(m,p;rho=0.5,seed=1,T=T)

# Change only this line to run the four new examples.
# family=SmoothQuantile(0.25,0.1)
family=Expectile(0.25)
# family=PseudoHuber(1.345)
# family=StudentT(4.0,1.0)

y,mu,eta=simulate_y_from_beta(X,beta_true,family;seed=seed,beta0=beta0_true)
Xfit=hcat(ones(T,m),X)

spectral=SpectralOptions{T}(k=5,krylovdim=12,ridge=0.0,resid_tol=5e-1,correction_tol=5e-2)
inner=InnerOptions{T}(solver=:pcg,maxiter=200,eta_max=0.6,forcing_c=10.0,forcing_alpha=0.0,abstol=1e-5,nesterov=true)
outer=OuterOptions{T}(maxiter=200,gtol=1e-6,relgtol=1e-8,nesterov=true,verbose=false)

results=benchmark_irls_methods(Xfit,y,family;spectral=spectral,inner=inner,outer=outer,
    ridge=0.0,maxiter=1000,gtol=1e-6,relgtol=1e-8,outer_nesterov=true,samples=5)
show(results,allrows=true,allcols=true)


7×8 DataFrame
 Row │ Method       Time_ms   Iter   Loss     GradNorm     RelGradNorm  Converged  LossGap     
     │ String       Float64   Int64  Float64  Float64      Float64      Bool       Float64     
─────┼─────────────────────────────────────────────────────────────────────────────────────────
   1 │ Spectral-MM   689.585     31  964.282  7.40644e-5   1.14057e-8        true  2.72848e-12
   2 │ LSMR          961.846      8  964.282  7.85204e-5   1.20919e-8        true  1.31877e-11
   3 │ CG            984.43       5  964.282  6.30181e-6   9.70458e-10       true  0.0
   4 │ LSQR         1049.84       8  964.282  5.74317e-5   8.8443e-9         true  3.63798e-12
   5 │ CGLS         1071.95       5  964.282  6.30181e-6   9.70458e-10       true  3.41061e-13
   6 │ CRLS         1918.59       5  964.282  5.71254e-6   8.79713e-10       true  2.27374e-13
   7 │ Cholesky     2788.9        5  964.282  1.70366e-11  2.62358e-15       true  0.0

In [8]:
seed=3124
p=5000; m=2p; T=Float64
beta_true=randn(MersenneTwister(seed),T,p)./sqrt(T(p))
beta0_true=T(0.5)
X=generate_correlated_design(m,p;rho=0.5,seed=1,T=T)

# Change only this line to run the four new examples.
# family=SmoothQuantile(0.25,0.1)
# family=Expectile(0.25)
family=PseudoHuber(1.345)
# family=StudentT(4.0,1.0)

y,mu,eta=simulate_y_from_beta(X,beta_true,family;seed=seed,beta0=beta0_true)
Xfit=hcat(ones(T,m),X)

spectral=SpectralOptions{T}(k=5,krylovdim=12,ridge=0.0,resid_tol=5e-1,correction_tol=5e-2)
inner=InnerOptions{T}(solver=:pcg,maxiter=200,eta_max=0.6,forcing_c=10.0,forcing_alpha=0.0,abstol=1e-5,nesterov=true)
outer=OuterOptions{T}(maxiter=200,gtol=1e-6,relgtol=1e-8,nesterov=true,verbose=false)

results=benchmark_irls_methods(Xfit,y,family;spectral=spectral,inner=inner,outer=outer,
    ridge=0.0,maxiter=1000,gtol=1e-6,relgtol=1e-8,outer_nesterov=true,samples=5)
show(results,allrows=true,allcols=true)


7×8 DataFrame
 Row │ Method       Time_ms   Iter   Loss     GradNorm     RelGradNorm  Converged  LossGap     
     │ String       Float64   Int64  Float64  Float64      Float64      Bool       Float64     
─────┼─────────────────────────────────────────────────────────────────────────────────────────
   1 │ Spectral-MM   696.339     29  4660.21  0.00028633   3.12402e-8        true  2.18279e-11
   2 │ CRLS         1892.01       9  4660.21  7.3135e-6    7.97944e-10       true  0.0
   3 │ CG           1991.05       9  4660.21  7.52847e-6   8.21399e-10       true  0.0
   4 │ CGLS         2095.14       9  4660.21  7.52847e-6   8.21399e-10       true  9.09495e-13
   5 │ LSQR         2125.01      12  4660.21  0.000561849  6.13009e-8        true  2.00089e-10
   6 │ LSMR         3718.91      18  4660.21  0.00025255   2.75546e-8        true  1.0823e-10
   7 │ Cholesky     5243.46       9  4660.21  2.09574e-6   2.28657e-10       true  0.0

In [9]:
seed=3124
p=5000; m=2p; T=Float64
beta_true=randn(MersenneTwister(seed),T,p)./sqrt(T(p))
beta0_true=T(0.5)
X=generate_correlated_design(m,p;rho=0.5,seed=1,T=T)

# Change only this line to run the four new examples.
# family=SmoothQuantile(0.25,0.1)
# family=Expectile(0.25)
# family=PseudoHuber(1.345)
family=StudentT(4.0,1.0)

y,mu,eta=simulate_y_from_beta(X,beta_true,family;seed=seed,beta0=beta0_true)
Xfit=hcat(ones(T,m),X)

spectral=SpectralOptions{T}(k=5,krylovdim=12,ridge=0.0,resid_tol=5e-1,correction_tol=5e-2)
inner=InnerOptions{T}(solver=:pcg,maxiter=200,eta_max=0.6,forcing_c=10.0,forcing_alpha=0.0,abstol=1e-5,nesterov=true)
outer=OuterOptions{T}(maxiter=200,gtol=1e-6,relgtol=1e-8,nesterov=true,verbose=false)

results=benchmark_irls_methods(Xfit,y,family;spectral=spectral,inner=inner,outer=outer,
    ridge=0.0,maxiter=1000,gtol=1e-6,relgtol=1e-8,outer_nesterov=true,samples=5)
show(results,allrows=true,allcols=true)


7×8 DataFrame
 Row │ Method       Time_ms    Iter   Loss     GradNorm     RelGradNorm  Converged  LossGap     
     │ String       Float64    Int64  Float64  Float64      Float64      Bool       Float64     
─────┼──────────────────────────────────────────────────────────────────────────────────────────
   1 │ Spectral-MM    854.928     46  4208.27  0.000214276   2.13039e-8       true  5.72982e-11
   2 │ LSMR          3216.07      25  4208.27  0.000248402   2.46967e-8       true  7.09406e-11
   3 │ LSQR          3295.8       23  4208.27  0.000508612   5.05675e-8       true  7.7307e-11
   4 │ CRLS          4671.6       25  4208.27  0.000103112   1.02517e-8       true  2.72848e-12
   5 │ CGLS          5268.81      24  4208.27  0.000213902   2.12667e-8       true  4.54747e-12
   6 │ CG            5334.81      24  4208.27  0.000213902   2.12667e-8       true  2.72848e-12
   7 │ Cholesky     14701.0       27  4208.27  0.000191065   1.89962e-8       true  0.0